# KS-test Parameter Grid: Precision/Recall/F1 vs. w_min, w_max, n_w, n_c, s_c, d_c (red/snr3)

Single-method, single-condition diagnostic: the augmented KS test (Bagniewski et al. 2021,
`ammonyte.utils.ks.KS_test` / `Series.kstest`), run on the **red/snr3** noise condition only
(`noise_sensitivity_data/noise_red_snr3.npz`, 20 seeds). Mirrors the same methodology as
`Pelt/pelt_grid_red_snr3.ipynb` / `Dynp/dynp_grid_red_snr3.ipynb` -- same data, same ground truth,
same train/test split, same `evaluate_detection` metric -- adapted to KS test's own parameters.

**No `cost` axis, no per-method loop.** Unlike Pelt/Dynp (9 `ruptures` cost functions), KS test is
one method with its own 6 free parameters, so this notebook runs a single joint grid search
instead of Section 7's "one cost per cell" pattern.

**Every parameter is in the grid -- none fixed by hand.** `w_min`, `w_max`, `n_w`, `n_c`, `s_c`,
`d_c` are all swept jointly and selected by mean F1 on the train pool, same discipline as
`pen`/`min_size` elsewhere: nothing is pre-judged as "not worth checking," and each parameter's
`on_edge` status is checked individually (Section 8) rather than assuming which ones matter.
`x_c` is the one exception, deliberately: `x_c=None` auto-computes as
`std(x)*0.1 + (max(x)-min(x))*0.05`, adapting to each seed's own noise scale. Gridding literal
`x_c` values across seeds with different noise realizations would silently break that
per-seed adaptivity and bias results toward whichever seeds happen to match the tested scale, so
`x_c` is left at its auto default for every combo rather than swept.

**No separate scoring script, no disk cache.** Matches the Pelt/binseg/BottomUp/Window/KernelCPD
convention (a single self-contained notebook, plain `joblib.Parallel`, no `joblib.Memory`) rather
than Dynp's `dynp_scoring.py` + disk cache -- that extra machinery existed specifically because
Dynp's O(n_bkps x n^2) DP cost made losing a partial run across SLURM/kernel sessions expensive.
KS test doesn't have that problem: a single call costs roughly `0.7s x n_w` (benchmarked directly
on this data), and the full grid below is sized to finish in one sitting.

**Grid sized to a ~7 hour / 24-CPU budget.** `w_min(4) x w_max(4) x n_w(3) x n_c(4) x s_c(4) x
d_c(7) = 5,376` combos x 14 train seeds = 75,264 scored calls, benchmarked at roughly ~6 hours on
24 cores (Section 8 re-estimates this live from one timed call before committing to the full run
-- check that estimate on the actual machine before launching). Ranges are grounded in this
dataset's own timing: `dt~=0.02`, ground-truth inter-transition gaps range 0.08-24.2 (min-max),
median 1.8 -- `w_min` stays near the floor needed for `n_c` points per window, `w_max` spans
below/above the median gap so a window doesn't routinely straddle two real transitions.

**Section 9 adds a per-parameter sensitivity check** (not present in the Pelt/Dynp notebooks,
since they only ever had 2 free axes per cost): for each of the 6 parameters, how much does mean
F1 actually move across its tested range, holding nothing else fixed -- answers "which parameters
are actually effective" rather than just "what's the single best combo."


In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ammonyte as amt
from ammonyte.utils.metrics import evaluate_detection
from joblib import Parallel, delayed

## Section 1 -- Configuration

In [ ]:
DATA_PATH   = "../../noise_sensitivity_data/noise_red_snr3.npz"
TOLERANCE   = 0.5   # +/- 0.5 kyr matching tolerance, matches cv_noise_sensitivity.py / Pelt / Dynp
RANDOM_SEED = 42
N_JOBS = -1   # CARC run: use all allocated cores; lower this if running locally

# Standard (non-heavy) seed budget -- same as Pelt/binseg/BottomUp/Window use for every cost
# function except rbf. KS test has no rbf-like outlier cost, so there's only one budget.
BUDGET = dict(pool_size=20, n_train=14, n_test=6)

# Grid -- every KS_test parameter except x_c (see intro markdown for why x_c is left at its
# per-seed auto default instead of gridded). Ranges grounded in this dataset's dt~=0.02 and
# ground-truth inter-transition gaps (min=0.08, median=1.8, max=24.2 kyr).
W_MIN_GRID = [0.05, 0.1, 0.2, 0.4]     # window-length floor: needs >= n_c*dt points to be valid
W_MAX_GRID = [0.5, 1, 2, 4]            # spans below/above the median 1.8 kyr gap
N_W_GRID   = [5, 10, 15]               # number of log-spaced window lengths between w_min/w_max
N_C_GRID   = [3, 5, 10, 15]            # minimum points required per window half
S_C_GRID   = [1.0, 1.5, 2.0, 2.5]      # std-ratio threshold
D_C_GRID   = [0.5, 0.6, 0.65, 0.7, 0.75, 0.8, 0.9]   # KS-statistic cutoff -- main sensitivity axis

PARAM_COLS = ['w_min', 'w_max', 'n_w', 'n_c', 's_c', 'd_c']
GRIDS = dict(w_min=W_MIN_GRID, w_max=W_MAX_GRID, n_w=N_W_GRID,
             n_c=N_C_GRID, s_c=S_C_GRID, d_c=D_C_GRID)

## Section 2 -- Data and ground truth

In [ ]:
def transition_times_from_states(time, states):
    """True transition times (both directions) from a state-label array.

    Mirrors the same helper in cv_noise_sensitivity.py / the Pelt/Dynp notebooks: any
    stadial/interstadial change counts as ground truth, regardless of direction.
    """
    change_idx = np.where(np.diff(states) != 0)[0] + 1
    return time[change_idx]


d = np.load(DATA_PATH)
time_arr   = d['time']
db_noisy   = d['db_noisy']
state_true = d['state_true']
n_seeds    = db_noisy.shape[0]

gt_times = [transition_times_from_states(time_arr, state_true[s]) for s in range(n_seeds)]
n_transitions = [len(g) for g in gt_times]
dt = float(np.median(np.diff(time_arr)))
print(f"Loaded {DATA_PATH}: {n_seeds} seeds, {len(time_arr)} points/series, dt~={dt:.4f}")
print(f"Ground-truth transitions per seed: min={min(n_transitions)}, max={max(n_transitions)}, "
      f"mean={np.mean(n_transitions):.1f}")

## Section 3 -- Detection wrapper and per-combo scoring

`x_c` is not part of `params` below (left at `Series.kstest`'s own auto default) -- see the intro
markdown for why.

In [ ]:
def _clean_times(times):
    times = np.asarray(times, dtype=float)
    return times[~np.isnan(times)]


def detect_kstest(time, value, w_min, w_max, n_w, n_c, s_c, d_c):
    series = amt.Series(time=time, value=value, label='grid', verbose=False)
    transitions = series.kstest(w_min=w_min, w_max=w_max, n_w=n_w, d_c=d_c, n_c=n_c, s_c=s_c, x_c=None)
    return _clean_times(transitions.jump_times)


def _score_one(params, seed):
    try:
        detected = detect_kstest(time_arr, db_noisy[seed], **params)
    except Exception:
        return dict(precision=0.0, recall=0.0, f1=0.0, false_positives=np.nan)
    m = evaluate_detection(detected=detected, ground_truth=gt_times[seed], tolerance=TOLERANCE)
    return dict(precision=m.precision, recall=m.recall, f1=m.f1_score, false_positives=m.false_positives)

## Section 4 -- Train/test split

One split for the whole notebook (no per-cost loop, so no `split_for_cost` needed) -- same
`RANDOM_SEED` and split logic as Pelt/Dynp, so seeds line up for cross-method comparison in
`compare_methods.ipynb`.

In [ ]:
def seed_pool(n_seeds_total, pool_size, random_state=RANDOM_SEED):
    """Fixed subsample of `pool_size` seed indices out of n_seeds_total."""
    rng = np.random.RandomState(random_state)
    return np.sort(rng.choice(n_seeds_total, size=pool_size, replace=False))


def outer_split(pool, n_train, n_test, random_state=RANDOM_SEED):
    """Split `pool` into disjoint train/test seed-index arrays, once."""
    rng = np.random.RandomState(random_state)
    shuffled = rng.permutation(pool)
    train_idx = np.sort(shuffled[:n_train])
    test_idx = np.sort(shuffled[n_train:n_train + n_test])
    return train_idx, test_idx


pool = seed_pool(n_seeds, BUDGET['pool_size'])
train_idx, test_idx = outer_split(pool, BUDGET['n_train'], BUDGET['n_test'])
print(f"train seeds ({len(train_idx)}): {train_idx}")
print(f"test seeds  ({len(test_idx)}): {test_idx}")

## Section 5 -- Grid search (train pool) + held-out evaluation (test pool) + d_c sweep

Same shared machinery as Pelt/Dynp's Section 5, generalized to `PARAM_COLS` (6 columns instead
of 2).

In [ ]:
def grid_search_train(param_grid, train_idx):
    """Score every (param, seed in train_idx) combo, aggregate per param, pick the
    winner by mean f1 over the train pool only."""
    jobs = [(pi, params, s) for pi, params in enumerate(param_grid) for s in train_idx]
    scored = Parallel(n_jobs=N_JOBS)(
        delayed(_score_one)(params, s) for pi, params, s in jobs
    )
    rows = []
    for (pi, params, s), res in zip(jobs, scored):
        row = dict(params)
        row.update(pi=pi, seed=int(s), split='train', **res)
        rows.append(row)
    train_raw = pd.DataFrame(rows)

    train_summary = (train_raw.groupby(PARAM_COLS)
                      .agg(mean_precision=('precision', 'mean'), std_precision=('precision', 'std'),
                           mean_recall=('recall', 'mean'), std_recall=('recall', 'std'),
                           mean_f1=('f1', 'mean'), std_f1=('f1', 'std'),
                           mean_fp=('false_positives', 'mean'))
                      .reset_index())

    best_row = train_summary.loc[train_summary['mean_f1'].idxmax()]
    best_params = {c: best_row[c] for c in PARAM_COLS}
    return best_params, best_row, train_summary, train_raw


def evaluate_test(best_params, test_idx):
    """Apply best_params to the untouched-until-now test pool, once."""
    scored = Parallel(n_jobs=N_JOBS)(
        delayed(_score_one)(best_params, s) for s in test_idx
    )
    rows = []
    for s, res in zip(test_idx, scored):
        row = dict(best_params)
        row.update(seed=int(s), split='test', **res)
        rows.append(row)
    test_raw = pd.DataFrame(rows)
    return (test_raw['precision'].mean(), test_raw['recall'].mean(),
            test_raw['f1'].mean(), test_raw['false_positives'].mean(), test_raw)


def d_c_sweep_test(fixed_params, d_c_grid, test_idx):
    """Sweep d_c at the winning w_min/w_max/n_w/n_c/s_c, test seeds only -- the
    precision/recall tradeoff curve for KS test (d_c is its PR-curve analog, same
    role pen/n_bkps play for Pelt/Dynp)."""
    other = {k: v for k, v in fixed_params.items() if k != 'd_c'}
    jobs = [(dc, s) for dc in d_c_grid for s in test_idx]
    scored = Parallel(n_jobs=N_JOBS)(
        delayed(_score_one)(dict(other, d_c=dc), s) for dc, s in jobs
    )
    rows = []
    for (dc, s), res in zip(jobs, scored):
        rows.append(dict(other, d_c=dc, seed=int(s), split='test_sweep', **res))
    sweep_raw = pd.DataFrame(rows)
    sweep_summary = (sweep_raw.groupby('d_c')
                      .agg(mean_precision=('precision', 'mean'), mean_recall=('recall', 'mean'),
                           mean_f1=('f1', 'mean'), mean_fp=('false_positives', 'mean'))
                      .reset_index())
    return sweep_summary, sweep_raw

## Section 6 -- Per-parameter sensitivity ("which parameters are actually effective")

Answers a different question than "what's the best combo": for each parameter, how much does
`mean_f1` move across its tested range once you average out every other parameter? A parameter
whose marginal mean barely moves isn't doing much (in the tested range); one with a clear trend
or peak is. This is a main-effects view -- cheap, no extra dependency, and it doesn't presuppose
which parameters matter before looking at the grid results.

In [ ]:
def param_main_effects(train_summary, param_cols, metric='mean_f1'):
    """Per-parameter marginal mean of `metric`, averaged over every other parameter.
    Returns {param_name: DataFrame(value, mean_metric)}."""
    effects = {}
    for p in param_cols:
        eff = (train_summary.groupby(p)[metric].mean()
               .reset_index().rename(columns={metric: f'mean_{metric}'}))
        effects[p] = eff
    return effects


def effect_size_ranking(effects, metric='mean_f1'):
    """Rank parameters by how much their marginal mean metric moves across the
    tested range (max - min of the main-effects curve) -- a quick, dependency-free
    stand-in for a full sensitivity/importance analysis."""
    col = f'mean_{metric}'
    rows = [dict(param=p, effect_range=eff[col].max() - eff[col].min())
            for p, eff in effects.items()]
    return pd.DataFrame(rows).sort_values('effect_range', ascending=False).reset_index(drop=True)

## Section 7 -- Plot helpers

In [ ]:
def plot_dc_sweep(sweep_summary, conclusion_row):
    """Test-set precision/recall tradeoff (d_c swept at the winning w_min/w_max/n_w/n_c/s_c)."""
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(sweep_summary['mean_recall'], sweep_summary['mean_precision'], color='#2a78d6',
             linewidth=1.5, marker='o', markersize=4, zorder=2)
    ax.scatter([conclusion_row['test_recall']], [conclusion_row['test_precision']], color='#e34948',
               s=90, marker='*', zorder=5, label='selected d_c')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel('recall')
    ax.set_ylabel('precision')
    ax.set_title("KS test: TEST precision/recall tradeoff (d_c swept)")
    ax.legend(frameon=False)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(color='#e1e0d9', linewidth=0.6)
    ax.set_axisbelow(True)
    plt.tight_layout()
    fig.savefig("ks_grid_red_snr3_dc_sweep.png", dpi=200, bbox_inches='tight')
    plt.show()


def plot_main_effects(effects, best_params, metric='mean_f1'):
    """One small subplot per parameter: its tested values on x, marginal mean_f1 on
    y (averaged over every other parameter) -- the main-effects view from Section 6."""
    col = f'mean_{metric}'
    params = list(effects.keys())
    ncols = 3
    nrows = int(np.ceil(len(params) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.4 * nrows), squeeze=False)

    for idx, p in enumerate(params):
        ax = axes[idx // ncols][idx % ncols]
        eff = effects[p]
        ax.plot(eff[p], eff[col], color='#2a78d6', linewidth=1.5, marker='o', markersize=4)
        ax.axvline(best_params[p], color='#e34948', linestyle='--', linewidth=1, alpha=0.7)
        ax.set_title(p, fontsize=10)
        ax.set_xlabel(p, fontsize=8)
        ax.set_ylabel(f'marginal {metric}', fontsize=8)
        ax.grid(color='#e1e0d9', linewidth=0.6)
        ax.set_axisbelow(True)

    for j in range(len(params), nrows * ncols):
        axes[j // ncols][j % ncols].axis('off')

    fig.suptitle("Per-parameter main effects (TRAIN grid, marginalized over the other 5 params)", y=1.0)
    plt.tight_layout()
    fig.savefig("ks_grid_red_snr3_main_effects.png", dpi=200, bbox_inches='tight')
    plt.show()

## Section 8 -- Run the grid

First cell estimates wall-clock time from one timed call on this machine before committing to
the full `Parallel` job below -- check this against your actual CARC allocation rather than
trusting the ~6h/24-core estimate from the intro, which was benchmarked on a different machine.

In [ ]:
param_grid = [dict(w_min=wn, w_max=wx, n_w=nw, n_c=nc, s_c=sc, d_c=dc)
              for wn in W_MIN_GRID for wx in W_MAX_GRID for nw in N_W_GRID
              for nc in N_C_GRID for sc in S_C_GRID for dc in D_C_GRID]

n_calls = len(param_grid) * len(train_idx)
print(f"Grid: {len(param_grid)} param combos x {len(train_idx)} train seeds = {n_calls} calls")

_mid = param_grid[len(param_grid) // 2]
_t0 = time.perf_counter()
_score_one(_mid, int(train_idx[0]))
_single_call_s = time.perf_counter() - _t0

_cpu_count = os.cpu_count() if N_JOBS == -1 else N_JOBS
_est_serial_h = n_calls * _single_call_s / 3600
_est_parallel_h = _est_serial_h / max(_cpu_count, 1)
print(f"Timed one call: {_single_call_s:.2f}s (params={_mid})")
print(f"Estimated: {_est_serial_h:.1f}h serial, ~{_est_parallel_h:.2f}h on {_cpu_count} cores "
      f"(assumes near-linear joblib scaling -- sanity check before trusting this)")

In [ ]:
t0 = time.perf_counter()
best_params, train_row, train_summary, train_raw = grid_search_train(param_grid, train_idx)
test_p, test_r, test_f1, test_fp, test_raw = evaluate_test(best_params, test_idx)
sweep_summary, sweep_raw = d_c_sweep_test(best_params, D_C_GRID, test_idx)

edges = [p for p in PARAM_COLS if best_params[p] in (min(GRIDS[p]), max(GRIDS[p]))]

conclusion_row = dict(
    method='KS_test',
    **{f'best_{p}': best_params[p] for p in PARAM_COLS},
    train_f1=train_row['mean_f1'], train_precision=train_row['mean_precision'],
    train_recall=train_row['mean_recall'],
    test_f1=test_f1, test_precision=test_p, test_recall=test_r, test_fp=test_fp,
    edge_params=",".join(edges) if edges else "none",
    n_train_seeds=len(train_idx), n_test_seeds=len(test_idx),
    train_seeds=",".join(map(str, train_idx)), test_seeds=",".join(map(str, test_idx)),
)

effects = param_main_effects(train_summary, PARAM_COLS)
ranking = effect_size_ranking(effects)

elapsed = time.perf_counter() - t0
edge_note = f"  <-- ON GRID EDGE: {edges}" if edges else ""
print(f"KS test: {best_params}  train_f1={train_row['mean_f1']:.3f}  test_f1={test_f1:.3f}  "
      f"({elapsed/3600:.2f}h){edge_note}")
print("\nEffect-size ranking (max-min of marginal mean_f1 across each param's tested range):")
print(ranking.to_string(index=False))

In [ ]:
plot_dc_sweep(sweep_summary, conclusion_row)
plot_main_effects(effects, best_params)

## Section 9 -- Save raw / summary / conclusion CSVs

In [ ]:
# raw: every scored (param, seed) row, tagged by split -- 'train' (full grid on the
# train pool, what selection used), 'test' (the single winning combo on the test
# pool), 'test_sweep' (d_c swept at the winning other params, for the PR-curve plot)
raw_df = pd.concat([train_raw, test_raw, sweep_raw], ignore_index=True, sort=False)
RAW_CSV = "ks_grid_red_snr3_raw.csv"
raw_df.to_csv(RAW_CSV, index=False)
print(f"Saved {len(raw_df)} rows to {RAW_CSV}")

SUMMARY_CSV = "ks_grid_red_snr3_summary.csv"
train_summary.to_csv(SUMMARY_CSV, index=False)
print(f"Saved {len(train_summary)} rows to {SUMMARY_CSV}")

CONCLUSION_CSV = "ks_grid_red_snr3_conclusion.csv"
pd.DataFrame([conclusion_row]).to_csv(CONCLUSION_CSV, index=False)

RANKING_CSV = "ks_grid_red_snr3_param_ranking.csv"
ranking.to_csv(RANKING_CSV, index=False)
print(f"Saved parameter effect-size ranking to {RANKING_CSV}")

In [ ]:
print("=" * 92)
print("CONCLUSION: KS test on red/snr3 -- train/test split, single joint grid over all 6 parameters")
print("=" * 92)
gap = conclusion_row['train_f1'] - conclusion_row['test_f1']
print(f"best params: { {p: best_params[p] for p in PARAM_COLS} }")
print(f"train_f1={conclusion_row['train_f1']:.3f}  test_f1={conclusion_row['test_f1']:.3f}  (gap={gap:+.3f})")
print(f"test_precision={conclusion_row['test_precision']:.3f}  test_recall={conclusion_row['test_recall']:.3f}")

if edges:
    print(f"\n{len(edges)} parameter(s) landed on a grid edge: {edges} -- widen those ranges and "
          f"rerun before trusting them as converged optima.")
else:
    print(f"\nAll {len(PARAM_COLS)} parameters converged (train optimum interior to their tested range).")

print(f"\nMost effective parameter (largest main-effect swing in mean_f1): "
      f"'{ranking.iloc[0]['param']}' (range={ranking.iloc[0]['effect_range']:.3f})")
print(f"Least effective: '{ranking.iloc[-1]['param']}' (range={ranking.iloc[-1]['effect_range']:.3f})")